In [4]:
from pyspark.sql import SparkSession
spark = (SparkSession.builder
         .master("local[4]")
         .appName("TP-RDD")
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("WARN")

import os
DATA = os.path.join(os.getcwd(), "data")   # chemin vu depuis le conteneur
os.makedirs(DATA, exist_ok=True)
print(DATA)

/home/jovyan/data


In [5]:
print(sc.master, sc.defaultParallelism, sc.version)

local[*] 8 4.0.0


In [6]:
citations = [
    "Spark est un moteur de calcul distribue",
    "Spark traite les donnees en memoire",
    "Hadoop stocke les donnees sur disque",
    "Spark est rapide et Spark est facile",
    "Les RDD sont immuables et distribues",
    "Le calcul distribue repose sur les RDD",
]
with open(DATA + "/citations.txt", "w") as f:
    f.write("\n".join(citations) + "\n")

In [7]:
x = sc.parallelize(range(1, 21), 4)
print(x.getNumPartitions())
print(x.glom().collect())
print(sc.parallelize(range(1, 21)).getNumPartitions())

txt = sc.textFile(DATA + "/citations.txt", 2)
print(txt.count(), txt.first(), type(txt))

4
[[1, 2, 3, 4, 5], [6, 7, 8, 9, 10], [11, 12, 13, 14, 15], [16, 17, 18, 19, 20]]
8
6 Spark est un moteur de calcul distribue <class 'pyspark.core.rdd.RDD'>


In [8]:
inexistant = sc.textFile(DATA + "/fichier_qui_n_existe_pas.txt")
print("RDD defini")

RDD defini


In [9]:
inexistant.count()

Py4JJavaError: An error occurred while calling z:org.apache.spark.api.python.PythonRDD.collectAndServe.
: org.apache.hadoop.mapred.InvalidInputException: Input path does not exist: file:/home/jovyan/data/fichier_qui_n_existe_pas.txt
	at org.apache.hadoop.mapred.FileInputFormat.singleThreadedListStatus(FileInputFormat.java:306)
	at org.apache.hadoop.mapred.FileInputFormat.listStatus(FileInputFormat.java:245)
	at org.apache.hadoop.mapred.FileInputFormat.getSplits(FileInputFormat.java:334)
	at org.apache.spark.rdd.HadoopRDD.getPartitions(HadoopRDD.scala:233)
	at org.apache.spark.rdd.RDD.$anonfun$partitions$2(RDD.scala:301)
	at scala.Option.getOrElse(Option.scala:201)
	at org.apache.spark.rdd.RDD.partitions(RDD.scala:297)
	at org.apache.spark.rdd.MapPartitionsRDD.getPartitions(MapPartitionsRDD.scala:49)
	at org.apache.spark.rdd.RDD.$anonfun$partitions$2(RDD.scala:301)
	at scala.Option.getOrElse(Option.scala:201)
	at org.apache.spark.rdd.RDD.partitions(RDD.scala:297)
	at org.apache.spark.ap

In [10]:
nombres = sc.parallelize([7, 10, 12, 17, 19], 2)
print(nombres.map(lambda a: a*a).collect())
print(nombres.map(lambda a: (a//2, a%2)).collect())
print(nombres.filter(lambda a: a%2 == 0).collect())
print(nombres.map(lambda a: a%2 == 0).collect())

p = sc.parallelize([(1, 2), (3, 5), (8, 10)])
print(p.flatMap(lambda t: t).collect())
print(p.map(lambda t: t).collect())

print(nombres.flatMap(lambda a: [a] if a%2 == 0 else []).collect())

z = sc.parallelize([1, 1, 2, 3, 4, 5, 5, 5, 6, 6])
print(z.distinct().collect())

[49, 100, 144, 289, 361]
[(3, 1), (5, 0), (6, 0), (8, 1), (9, 1)]
[10, 12]
[False, True, True, False, False]
[1, 2, 3, 5, 8, 10]
[(1, 2), (3, 5), (8, 10)]
[10, 12]
[1, 2, 3, 4, 5, 6]


In [11]:
y = sc.parallelize([6, 8, 9, 1, 2, 3], 3)
print(y.count(), y.first(), y.take(3), y.take(100), y.takeOrdered(3), y.collect())

print(y.reduce(lambda a, b: a + b), y.sum(), y.max(), y.min(), y.mean())

vide = sc.parallelize([])
print(vide.take(3))
try:
    vide.first()
except ValueError as e:
    print("Erreur :", e)
print(vide.isEmpty())

6 6 [6, 8, 9] [6, 8, 9, 1, 2, 3] [1, 2, 3] [6, 8, 9, 1, 2, 3]
29 29 9 1 4.833333333333333
[]
Erreur : RDD is empty
True


In [12]:
words = txt.map(lambda l: l.lower()).flatMap(lambda l: l.split(" "))
print(words.count(), words.distinct().count())      # 39, 25

wc = words.map(lambda w: (w, 1)).reduceByKey(lambda a, b: a + b)
print(wc.takeOrdered(5, key=lambda kv: -kv[1]))

print(txt.filter(lambda l: "Spark" in l).count())    # 3
print(txt.map(lambda l: len(l.split(" "))).max())    # 7

import shutil
shutil.rmtree(DATA + "/wc_out", ignore_errors=True)
wc.coalesce(1).saveAsTextFile(DATA + "/wc_out")
print(os.listdir(DATA + "/wc_out"))
print(sc.textFile(DATA + "/wc_out").count())         # 25

39 25
[('les', 4), ('spark', 4), ('est', 3), ('distribue', 2), ('donnees', 2)]
3
7
['part-00000', '._SUCCESS.crc', '_SUCCESS', '.part-00000.crc']
25


In [13]:
wc = words.map(lambda w: (w, 1)).reduceByKey(lambda a, b: a + b)
print(wc.collect())


[('est', 3), ('un', 1), ('distribue', 2), ('traite', 1), ('les', 4), ('donnees', 2), ('hadoop', 1), ('sur', 2), ('rapide', 1), ('et', 2), ('rdd', 2), ('sont', 1), ('immuables', 1), ('repose', 1), ('spark', 4), ('moteur', 1), ('de', 1), ('calcul', 2), ('en', 1), ('memoire', 1), ('stocke', 1), ('disque', 1), ('facile', 1), ('distribues', 1), ('le', 1)]


In [14]:
import shutil
shutil.rmtree(DATA + "/wc_out", ignore_errors=True)
wc.coalesce(1).saveAsTextFile(DATA + "/wc_out")
print(os.listdir(DATA + "/wc_out"))
print(sc.textFile(DATA + "/wc_out").count())
 
# Relance sans supprimer le dossier :
wc.coalesce(1).saveAsTextFile(DATA + "/wc_out")


['part-00000', '._SUCCESS.crc', '_SUCCESS', '.part-00000.crc']
25


Py4JJavaError: An error occurred while calling o590.saveAsTextFile.
: org.apache.hadoop.mapred.FileAlreadyExistsException: Output directory file:/home/jovyan/data/wc_out already exists
	at org.apache.hadoop.mapred.FileOutputFormat.checkOutputSpecs(FileOutputFormat.java:131)
	at org.apache.spark.internal.io.HadoopMapRedWriteConfigUtil.assertConf(SparkHadoopWriter.scala:303)
	at org.apache.spark.internal.io.SparkHadoopWriter$.write(SparkHadoopWriter.scala:73)
	at org.apache.spark.rdd.PairRDDFunctions.$anonfun$saveAsHadoopDataset$1(PairRDDFunctions.scala:1094)
	at scala.runtime.java8.JFunction0$mcV$sp.apply(JFunction0$mcV$sp.scala:18)
	at org.apache.spark.rdd.RDDOperationScope$.withScope(RDDOperationScope.scala:151)
	at org.apache.spark.rdd.RDDOperationScope$.withScope(RDDOperationScope.scala:112)
	at org.apache.spark.rdd.RDD.withScope(RDD.scala:417)
	at org.apache.spark.rdd.PairRDDFunctions.saveAsHadoopDataset(PairRDDFunctions.scala:1092)
	at org.apache.spark.rdd.PairRDDFunctions.$anonfu